# Lesson 11｜圖片與檔案下載

前面抓的都是「文字」：HTML、JSON。今天抓「檔案」：圖片、CSV、大型檔案。
它們不是文字，而是**二進位資料（bytes）**，抓法和存法都要稍微不同。抓回來之後，用 Pillow 縮圖、轉格式、整理成資料夾。
今天分三段：

- **A. 下載圖片**：圖片是 bytes、找出圖片網址、單張與批次下載（還要避開有問題的圖片）
- **B. 大檔案與其他檔案**：串流下載、先問大小再下載、CSV 檔案
- **C. 用 Pillow 處理圖片**：看資訊、縮圖、轉格式與透明、依評分整理資料夾、做縮圖牆

在 VS Code 按每格上方的 Run Cell 逐格執行。


### Notebook 使用方式

先選擇課程 Python 環境，再由上到下執行每一格（Shift + Enter）。先執行工作資料夾設定及前面的匯入。
若看到 `NameError`，先確認變數名稱，並回到上方執行建立該變數的格子；必要時重新啟動核心並從頭執行。練習或作業格要先完成 TODO，再執行檢查。


## 設定工作資料夾

請先執行下一格。使用 VS Code 開啟完整教材資料夾；本堂的 `fixtures/` 和 `output/` 會以本堂資料夾為基準。


In [ ]:
import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson11_images_and_files'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())

import sys
tools_path = str(lesson_path.parents[1] / 'tools')
if tools_path not in sys.path:
    sys.path.insert(0, tools_path)


In [ ]:
import io
import mimetypes
import re
import shutil
import time
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageOps, UnidentifiedImageError

from practice_site import start_server, stop_server

# 本機練習站：/gallery 是圖庫頁，/images/ 放圖片，/files/ 放要下載的檔案
server, BASE_URL = start_server()
GALLERY = Path("fixtures/gallery")  # 同一批圖片的本機副本，第 08 單元之後會用到
Path("output").mkdir(exist_ok=True)
print("本機練習站：", BASE_URL, "（用瀏覽器打開 /gallery 看看）")


## 01｜今天的成果預覽

抓一張書的封面，存成檔案，再用 Pillow 做成縮圖。


In [ ]:
response = requests.get(f"{BASE_URL}/images/cover_01.jpg", timeout=10)
cover_path = Path("output/cover_01.jpg")
cover_path.write_bytes(response.content)

with Image.open(cover_path) as cover:
    print("原圖：", cover.size, cover.format, cover_path.stat().st_size, "bytes")
    thumb = ImageOps.fit(cover, (120, 160))  # 裁切並縮成剛好 120 × 160
    thumb.save("output/cover_01_thumb.jpg")
print("縮圖：", Image.open("output/cover_01_thumb.jpg").size, Path("output/cover_01_thumb.jpg").stat().st_size, "bytes")


# A. 下載圖片

## 02｜圖片是「二進位資料」

文字用 `response.text`，檔案用 `response.content`（bytes）。
存檔時要用「二進位寫入」：`Path.write_bytes(...)`，或 `open(path, "wb")`（`b` 是 binary）。


In [ ]:
response = requests.get(f"{BASE_URL}/images/cover_01.jpg", timeout=10)
print("狀態碼：", response.status_code)
print("Content-Type：", response.headers["Content-Type"])  # 內容是什麼類型
print("Content-Length：", response.headers["Content-Length"])  # 檔案多大（位元組）
print("content 的型別：", type(response.content), "｜長度：", len(response.content))
print("前 4 個位元組：", response.content[:4], "（JPEG 檔案一定以 ff d8 開頭）")
print("提醒：圖片不要用 response.text，那會把二進位硬當成文字解碼，只會得到亂碼。")


### HEAD：只問標頭，不下載內容

`requests.head(url)` 回傳和 `get` 一樣的標頭，但沒有內容。下載大檔案前，可以先問「多大？是什麼？」再決定要不要抓。


In [ ]:
head = requests.head(f"{BASE_URL}/images/cover_01.jpg", timeout=10)
print("HEAD：", head.status_code, head.headers["Content-Type"], head.headers["Content-Length"], "｜內容長度：", len(head.content))


## 03｜從網頁找出圖片網址

圖片網址在 `<img>` 的 `src` 屬性。但是：

1. `src` 常常是**相對路徑**（`images/cover_01.jpg`），要用 `urljoin` 接上頁面網址。
2. 有些網站用「懶載入」：`src` 只是佔位的小圖，真正的網址放在 `data-src`。
3. 網址後面可能有參數（`?v=3`）。


In [ ]:
page_url = f"{BASE_URL}/gallery"
soup = BeautifulSoup(requests.get(page_url, timeout=10).text, "html.parser")
images = soup.select("img")
print("頁面上共有", len(images), "個 <img>")
for img in images[:3] + images[-4:]:
    print(f"  src={img['src']!r:32} data-src={img.get('data-src')!r}  alt={img['alt'][:20]!r}")


In [ ]:
# 一般圖片：src 接上頁面網址
print(urljoin(page_url, "images/cover_01.jpg"))
print(urljoin(page_url, "/images/cover_01.jpg"))  # 開頭有 / 代表從網站根目錄算起

# 懶載入的圖片：優先用 data-src
lazy = soup.select_one("img.lazy")
print("懶載入的 src ：", urljoin(page_url, lazy["src"]))
print("懶載入的 data-src：", urljoin(page_url, lazy["data-src"]))

image_urls = [urljoin(page_url, img.get("data-src") or img["src"]) for img in images]  # data-src 優先
print(len(image_urls), "個真正的圖片網址；最後 3 個：", [url.rsplit("/", 1)[-1] for url in image_urls[-3:]])


## 04｜從網址取出檔名

不要用 `url.split("/")[-1]`：網址有 `?v=3` 參數時，檔名會跟著帶進去。
先用 `urlparse(url).path` 取出「路徑」，再取最後一段。


In [ ]:
url = f"{BASE_URL}/images/cover_02.jpg?v=3"
print("split 的結果：", url.split("/")[-1])  # 檔名多了 ?v=3
print("urlparse 的結果：", Path(urlparse(url).path).name)  # cover_02.jpg


def filename_from_url(url):
    return Path(urlparse(url).path).name


## 05｜單張下載函式

圖庫裡有兩張「有問題的圖」：

- `missing.jpg`：網站回 **404**（找不到）。
- `fake.jpg`：網站回 **200**（成功），但內容是一個錯誤網頁，不是圖片！

所以不能只看狀態碼，還要檢查 `Content-Type` 是不是以 `image/` 開頭。


In [ ]:
def download_image(url, folder):
    """下載一張圖片，回傳存好的 Path；失敗回傳 None。"""
    try:
        response = requests.get(url, timeout=10)
    except requests.RequestException as error:
        print(f"  連線失敗：{url} → {type(error).__name__}")
        return None
    content_type = response.headers.get("Content-Type", "")
    if response.status_code != 200:
        print(f"  略過：{filename_from_url(url)} 狀態碼 {response.status_code}")
        return None
    if not content_type.startswith("image/"):
        print(f"  略過：{filename_from_url(url)} 不是圖片（Content-Type：{content_type}）")
        return None
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename_from_url(url)
    path.write_bytes(response.content)
    return path


for name in ("cover_03.jpg", "missing.jpg", "fake.jpg"):
    print(name, "→", download_image(f"{BASE_URL}/images/{name}", "output/single"))


## 06｜批次下載

把網址一個一個丟進去。三個好習慣：

1. **檔案已經存在就略過**：中途失敗、重新執行時，不用從頭再抓。
2. **每張之間 `time.sleep`**：對真實網站要有禮貌。練習站用 0.05 秒，真實網站請 1 秒以上。
3. **最後印統計**：成功幾張、略過幾張、失敗幾張。


In [ ]:
def download_all(urls, folder, delay=1.0):
    summary = {"ok": 0, "skipped": 0, "failed": 0}
    for url in urls:
        if (Path(folder) / filename_from_url(url)).exists():
            summary["skipped"] += 1
            continue
        if download_image(url, folder) is None:
            summary["failed"] += 1
        else:
            summary["ok"] += 1
        time.sleep(delay)
    return summary


shutil.rmtree("output/images", ignore_errors=True)
print("第一次：", download_all(image_urls, "output/images", delay=0.05))
print("第二次：", download_all(image_urls, "output/images", delay=0.05), "← 全部略過，沒有重抓")
total = sum(path.stat().st_size for path in Path("output/images").iterdir())
print(f"共 {len(list(Path('output/images').iterdir()))} 個檔案，{total / 1024:.0f} KB")


# B. 大檔案與其他檔案

## 07｜大檔案：串流下載

`requests.get(url)` 會把整個檔案先讀進記憶體，才給你。檔案很大（幾百 MB）時會吃光記憶體。
加上 `stream=True`，再用 `iter_content(chunk_size)` 一塊一塊讀、一塊一塊寫，記憶體只需要一塊的大小。


In [ ]:
def download_file(url, path, chunk_size=64 * 1024):
    """串流下載，回傳實際下載的位元組數；過程中每 25% 印一次進度。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=10) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))
        done, next_mark = 0, 25
        with open(path, "wb") as file:
            for chunk in response.iter_content(chunk_size):
                file.write(chunk)
                done += len(chunk)
                if total and done * 100 // total >= next_mark:
                    print(f"  {done * 100 // total}%（{done:,} / {total:,} bytes）")
                    next_mark += 25
    return done


size = download_file(f"{BASE_URL}/files/big.bin", "output/big.bin")
print("下載完成：", size, "bytes ｜檔案大小：", Path("output/big.bin").stat().st_size)


## 08｜先問再下載，與下載 CSV

連結的 `Content-Disposition` 標頭常常會告訴你「建議的檔名」。CSV 是文字，下載後可以直接用 pandas 讀。


In [ ]:
csv_url = f"{BASE_URL}/files/price_list.csv"
head = requests.head(csv_url, timeout=10)
print("類型：", head.headers["Content-Type"])
print("大小：", head.headers["Content-Length"], "bytes")
print("建議檔名：", head.headers["Content-Disposition"])

filename = re.search(r'filename="([^"]+)"', head.headers["Content-Disposition"]).group(1)
csv_path = Path("output") / filename
download_file(csv_url, csv_path)
print(pd.read_csv(csv_path).to_string(index=False))


# C. 用 Pillow 處理圖片

Pillow（`from PIL import Image`）是 Python 最常用的圖片套件。後面都用 `fixtures/gallery`（和練習站同一批圖）。

## 09｜開啟圖片，看資訊

- `Image.open(path)`：開啟（用 `with`，用完自動關檔）
- `.size`：`(寬, 高)`　`.format`：`JPEG`、`PNG`　`.mode`：`RGB`（彩色）、`RGBA`（彩色加透明）、`L`（灰階）


In [ ]:
rows = []
for path in sorted(GALLERY.glob("*.[jp][pn]g")):
    with Image.open(path) as picture:
        rows.append(
            {"file": path.name, "width": picture.width, "height": picture.height, "format": picture.format,
             "mode": picture.mode, "kb": round(path.stat().st_size / 1024, 1)}
        )
info = pd.DataFrame(rows)
print(info.to_string(index=False))


### 不是圖片的檔案，Pillow 打不開

還記得 `fake.jpg` 嗎？如果沒有檢查 `Content-Type` 就存了檔，之後 Pillow 開檔會出錯。


In [ ]:
fake_bytes = requests.get(f"{BASE_URL}/images/fake.jpg", timeout=10).content
try:
    Image.open(io.BytesIO(fake_bytes))  # io.BytesIO：把 bytes 當成檔案來開，不用先存檔
except UnidentifiedImageError as error:
    print("UnidentifiedImageError：", error)
print("→ 下載時就先檢查 Content-Type，比事後處理壞檔簡單。")


## 10｜縮圖：`thumbnail`、`resize`、`ImageOps.fit`

| 方法 | 作法 | 結果 |
|---|---|---|
| `thumbnail((w, h))` | 在原圖上依比例縮小，放進 w × h 的框裡 | 比例不變，**可能比框小**；不會放大 |
| `resize((w, h))` | 硬拉成 w × h | 剛好 w × h，但**會變形** |
| `ImageOps.fit(圖, (w, h))` | 依比例縮放再從中間裁切 | 剛好 w × h，**不變形**，但邊緣會被切掉 |


In [ ]:
with Image.open(GALLERY / "cover_01.jpg") as cover:
    print("原圖：", cover.size)

    shrunk = cover.copy()  # thumbnail 會直接改動圖片本身，所以先複製
    shrunk.thumbnail((120, 160))
    stretched = cover.resize((120, 160))
    fitted = ImageOps.fit(cover, (120, 160))
    print("thumbnail：", shrunk.size, "｜resize：", stretched.size, "｜fit：", fitted.size)

    for label, picture in (("thumbnail", shrunk), ("resize", stretched), ("fit", fitted)):
        picture.save(f"output/cover_01_{label}.jpg")
print("打開 output/ 裡的三張圖比較看看：resize 的封面被壓扁了。")


### JPEG 品質與檔案大小

存 JPEG 時的 `quality`（1–95，預設 75）越低，檔案越小、畫質越差。網頁用途 80–85 通常看不出差別。


In [ ]:
with Image.open(GALLERY / "landscape.jpg") as picture:
    for quality in (95, 85, 70, 40):
        buffer = io.BytesIO()
        picture.save(buffer, "JPEG", quality=quality)
        print(f"quality={quality}：{buffer.tell() / 1024:.1f} KB")


## 11｜轉格式與透明

PNG 可以有「透明」（RGBA 模式），JPEG 不行。直接把 RGBA 存成 JPEG 會出錯。
解法：先貼到白色（或你想要的顏色）背景上，變成 RGB 再存。
（直接 `convert("RGB")` 也能存，但透明的地方會變成**黑色**。）


In [ ]:
banner = Image.open(GALLERY / "banner.png")
print("模式：", banner.mode, "｜左上角像素（透明）：", banner.getpixel((0, 0)))

try:
    banner.save("output/banner_error.jpg")
except OSError as error:
    print("OSError：", error)

black = banner.convert("RGB")
print("convert('RGB') 後的左上角：", black.getpixel((0, 0)), "← 透明變成黑色")

background = Image.new("RGB", banner.size, "white")
background.paste(banner, mask=banner.getchannel("A"))  # 用透明通道當遮罩
print("貼到白底後的左上角：", background.getpixel((0, 0)))
background.save("output/banner.jpg", quality=85)
print("已存檔：output/banner.jpg", Path("output/banner.jpg").stat().st_size, "bytes")


## 12｜依評分整理資料夾

目標：把 12 本書的封面做成 120 × 160 的縮圖，依評分放進資料夾：

```
output/by_rating/Five/05_sapiens-a-brief-history-of-humankind.jpg
output/by_rating/Four/04_sharp-objects.jpg
...
```

檔名要「安全」：Windows 不允許 `\ / : * ? " < > |`，書名又常有空白與標點。
所以用 `slugify`：轉小寫，連續的非英數字元換成一個 `-`。


In [ ]:
def slugify(title):
    slug = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")
    return slug[:40].rstrip("-")  # 太長的書名只留前 40 個字元


for title in ("Sapiens: A Brief History of Humankind", "Starving Hearts (Triangular Trade Trilogy, #1)"):
    print(f"{title!r} → {slugify(title)!r}")


def make_thumb(src, dst, size=(120, 160)):
    """裁切縮成剛好 size，有透明就貼白底，存成 JPEG。"""
    with Image.open(src) as picture:
        if picture.mode == "RGBA":
            background = Image.new("RGB", picture.size, "white")
            background.paste(picture, mask=picture.getchannel("A"))
            picture = background
        thumb = ImageOps.fit(picture.convert("RGB"), size)
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    thumb.save(dst, "JPEG", quality=85)
    return dst


In [ ]:
# 書名與評分在圖庫頁的 <figure class="book"> 裡：data-rating 是評分，<figcaption> 是書名
books = []
for figure in soup.select("figure.book"):
    img = figure.select_one("img")
    books.append(
        {
            "file": filename_from_url(urljoin(page_url, img.get("data-src") or img["src"])),
            "title": figure.select_one("figcaption").get_text(),
            "rating": figure["data-rating"],
        }
    )
print(len(books), "本書；第一本：", books[0])

shutil.rmtree("output/by_rating", ignore_errors=True)
for number, book in enumerate(books, start=1):
    name = f"{number:02d}_{slugify(book['title'])}.jpg"
    make_thumb(GALLERY / book["file"], Path("output/by_rating") / book["rating"] / name)

for folder in sorted(Path("output/by_rating").iterdir()):
    print(f"{folder.name:6}", [path.name[:26] for path in sorted(folder.iterdir())])


## 13｜縮圖牆

把很多張圖排成一張大圖（contact sheet），一眼看完整批資料。作法：

1. `Image.new("RGB", (寬, 高), "white")` 建立一張白色底圖
2. 算出每一格的位置 `(x, y)`，用 `sheet.paste(小圖, (x, y))` 貼上去


In [ ]:
def contact_sheet(paths, cols=4, cell=(120, 160), gap=8):
    rows = -(-len(paths) // cols)  # 無條件進位
    width = cols * cell[0] + (cols + 1) * gap
    height = rows * cell[1] + (rows + 1) * gap
    sheet = Image.new("RGB", (width, height), "white")
    for index, path in enumerate(paths):
        with Image.open(path) as picture:
            tile = ImageOps.fit(picture.convert("RGB"), cell)
        x = gap + (index % cols) * (cell[0] + gap)
        y = gap + (index // cols) * (cell[1] + gap)
        sheet.paste(tile, (x, y))
    return sheet


wall = contact_sheet([GALLERY / book["file"] for book in books])
wall.save("output/wall.jpg", quality=90)
print("縮圖牆：", wall.size, "→ 打開 output/wall.jpg 看看")


## 14｜常見問題與守則

- **檔名重複**：不同網址可能有同樣的檔名（`1.jpg`）。可以加上編號、或把網址的雜湊值當檔名。
- **網址沒有副檔名**：用 `Content-Type` 推測，`mimetypes.guess_extension("image/jpeg")`。
- **Windows 的非法檔名字元**：`\ / : * ? " < > |` 要換掉。
- **RGBA 存 JPEG 出錯**：先貼到白底變 RGB（單元 11）。
- **手機照片轉向不對**：照片的方向放在 EXIF 裡，可以用 `ImageOps.exif_transpose(圖)` 修正。
- **磁碟空間與流量**：圖片檔很大。先用 HEAD 看大小、設定上限、控制數量與頻率。
- **版權**：自己下載來分析、學習可以；重新發布別人的圖片通常不行。課堂用的是練習網站的圖片。


In [ ]:
print(mimetypes.guess_extension("image/jpeg"), mimetypes.guess_extension("image/png"))
print(re.sub(r'[\\/:*?"<>|]', "_", 'report: 2026/10 "final"?.jpg'))


## 收尾：關閉練習站

今天下課前，確認 `output/` 裡有：`by_rating/` 資料夾、`wall.jpg`、`images/`、`big.bin`、`price_list.csv`。


In [ ]:
stop_server(server)
print("練習站已關閉")
